## Phase 0: Configuration and Field Mapping

In [296]:
### CONFIGURATION & DEMO VARIABLES ###

# Set the path for your input CSV files
loinc_csv_path = 'SmallTestCSVs/Loinc.csv'
part_link_csv_path = 'SmallTestCSVs/Part.csv'
answer_list_csv_path = 'SmallTestCSVs/AnswerList.csv'
linguistic_variants_path = 'SmallTestCSVs/LinguisticVariants'

# Set the output path for the transformed JSONL file
output_folder = 'output'

# Set this to 1 or 2 to run the notebook with example data instead of your real CSVs.
# NOTE: The demo data below is for illustration; real-world data is much larger.
mode = 0  # 0 = full run with all data, 1 = test_mode with real subset of data up to 15 records per CSV, 2 = demo_mode with example data

In [297]:
### PACKAGE IMPORTS ###

# Import pandas for data manipulation and analysis
import pandas as pd

# Import numpy for numerical operations, often used for NaN values
import numpy as np

# Import json for saving to JSON Lines format
import json

#Import StringIO to handle in-memory text streams
from io import StringIO

# Import os for file and directory operations
import os

In [298]:
# Example DataFrames for demo_mode (mode = 2)
# These are based on the CSV snippets you provided.
demo_loinc_df = pd.DataFrame({
    'LOINC_NUM': ['100000-9', '100001-7'],
    'COMPONENT': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'PROPERTY': ['Hx', 'LP431396-3'],
    'TIME_ASPCT': ['Pt', 'Pt'],
    'SYSTEM': ['^Patient', 'Ser'],
    'SCALE_TYP': ['Nar', 'Qn'],
    'SHORTNAME': ['Health Info Pioneer+Father of LOINC', 'Health Info Pioneer+Cofounder of LOINC'],
    'LONG_COMMON_NAME': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'STATUS': ['ACTIVE', 'ACTIVE']
})

demo_part_link_df = pd.DataFrame({
    'LoincNumber': ['100000-9', '100000-9', '100000-9', '100000-9', '100000-9'],
    'LongCommonName': ['Health informatics pioneer and the father of LOINC'] * 5,
    'PartNumber': ['LP431397-1', 'LP6817-3', 'LP6960-1', 'LP310005-6', 'LP7749-7'],
    'PartName': ['Health informatics pioneer and the father of LOINC', 'Hx', 'Pt', '^Patient', 'Nar'],
    'PartCodeSystem': ['http://loinc.org'] * 5,
    'PartTypeName': ['COMPONENT', 'PROPERTY', 'TIME', 'SYSTEM', 'SCALE'],
    'LinkTypeName': ['Primary'] * 5,
    'Property': ['http://loinc.org/property/COMPONENT', 'http://loinc.org/property/PROPERTY', 'http://loinc.org/property/TIME_ASPCT', 'http://loinc.org/property/SYSTEM', 'http://loinc.org/property/SCALE_TYP']
})

demo_answer_list_df = pd.DataFrame({
    'AnswerListId': ['LL1000-0', 'LL1000-0', 'LL1000-0', 'LL1001-8'],
    'AnswerListName': ['PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_14_30D freq amts'],
    'AnswerListOID': ['1.3.6.1.4.1.12009.10.1.165'] * 3 + ['1.3.6.1.4.1.12009.10.1.166'],
    'ExtDefinedYN': ['N'] * 4,
    'AnswerStringId': ['LA13825-7', 'LA13838-0', 'LA13892-7', 'LA6270-8'],
    'SequenceNumber': [1, 2, 3, 1],
    'DisplayText': ['1 slice or 1 dinner roll', '2 slices or 2 dinner rolls', 'More than 2 slices or 2 dinner rolls', 'Never']
})

# Linguistic variant CSV demo data 
esMX_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
33512-5,Color,Tipo,Punto temporal,XXX,Nominal,,,,Color: XXX : Punto temporal: Tipo: Nominal:,,
24355-0,Panel macroscópico de análisis de orina,-,Punto temporal,Orina,-,,,,Panel macroscópico de análisis de orina: Orina : Punto temporal: -: -:,,
10003-2,Duración de la onda R. derivación III,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación III:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10006-5,Duración de la onda R. derivación V3,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V3:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10007-3,Duración de la onda R. derivación V4,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V4:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10024-8,Duración de la onda R 'plomo AVR,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R 'plomo AVR:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
1004-1,"Test de antiglobulina directo, reactivo específico del complemento",Presencia o umbral,Punto temporal,Eritrocitos,Ordinal,,,,"Test de antiglobulina directo, reactivo específico del complemento: Eritrocitos : Punto temporal: Presencia o umbral: Ordinal:",,
10060-2,Amplitud de onda S. Conduzca AVR,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. Conduzca AVR:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10063-6,Amplitud de onda S. derivación III,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. derivación III:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10280-6,Número de modelo del proveedor,Tipo,Punto temporal,Tubo de cobre,Nominal,,,,Número de modelo del proveedor:Tubo de cobre :Punto temporal:Tipo:Nominal:,,
10445-5,CD11c Ag,Presencia o umbral,Punto temporal,Tejido y frotis,Ordinal,Mancha inmune,,,CD11c Ag: Tejido y frotis : Punto temporal: Presencia o umbral: Ordinal: Mancha inmune,,
10455-4,Xilosa ^30 M después de 25 g de xilosa VO,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Xilosa : Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10539-5,glipizida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,glipizida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10547-8,Primidona + FENobarbital,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Primidona + FENobarbital: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10550-2,Temazepam,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Temazepam: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
1099-1,K sub p super sub a Ab,Presencia o umbral,Punto temporal,Suero o Plasma,Ordinal,,,,K sub p super sub a Ab: Suero o Plasma : Punto temporal: Presencia o umbral: Ordinal:,,
10995-9,Neomicina,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Neomicina: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
11001-5,Pirazinamida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Pirazinamida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,"""

etEE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
93488-5,Guanidinoatsetaat,SCnc,Pt,Vereplekk,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Veri,
93505-6,Heptakarboksüülporfüriin I,SRat,24 tunni,U,Qn,,CHEM,,,Aine määr Kvantitatiivne Uriin,
93729-2,Beeta-2-mikroglobuliin/kreatiniin,Suhe,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
93748-2,Fibriini monomeerid,MCnc,Pt,PPP,Qn,IA,COAG,,,Juhuslik Kvantitatiivne Trombotsüütidevaene plasma,
95073-3,Histoplasma capsulatum antigeen,MCnc,Pt,BalF,Qn,IA,MICRO,,,Juhuslik Kvantitatiivne,
95074-1,Bakterid,PrThr,Pt,BalF,Ord,Valgusmikroskoopia,MICRO,,,Järgarvuline Juhuslik,
89481-6,Gentamütsiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92255-9,Metitsilliin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92242-7,Pürasiinamiid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96635-8,HLA-C,Tüüp,Pt,B/Tis^doonor,Nom,,HLA,,,Juhuslik Kude Veri Veri või koematerjal,
95563-3,16-alfahüdroksüdehüdroepiandrosteroon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95593-0,25-hüdroksükaltsiferool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95114-5,Insuliin^2 tundi pärast sööki,Acnc,Pt,S/P,Qn,,CHAL,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
93773-0,11-deoksükortisool,SCnc,Pt,Sal,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Sülg,
93838-1,Histoplasma capsulatum antikehad.IgM,Acnc,Pt,CSF,Qn,IA,MICRO,,,Immuunglobuliin M Juhuslik Kvantitatiivne Liikvor,
94255-7,Kaltsium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94256-5,Magneesium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94270-6,Ubikinoon 10,SCnt,Pt,WBC,Qn,,CHEM,,,Ainehulga sisaldus Juhuslik Kvantitatiivne Leukotsüüdid,
95543-5,Aspergillus terreus antikehad.IgG,PrThr,Pt,S,Ord,,ALLERGY,,,Immuunglobuliin G Järgarvuline Juhuslik Seerum,
95527-8,Tsütomegaloviirus antikehad.IgG,PrThr,Pt,Sal,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Sülg,
95688-8,Dengue viiruse 1.+ 2.+ 3.+ 4. tüüp antikehad.IgM,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin M Järgarvuline Juhuslik Täpsustamata materjal,
93771-4,Kalprotektiin,MCnc,Pt,SynF,Qn,,CHEM,,,Juhuslik Kvantitatiivne Liigesevedelik sünoviaalvedelik,
95574-0,17-alfahüdroksüpregnanoloon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95594-8,Kaltsidiool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95675-5,Kollapalaviku viirus antikehad.IgG,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Täpsustamata materjal,
95719-1,Flaviviirus antikehad,PrThr,Pt,S/P,Ord,,MICRO,,,Järgarvuline Juhuslik Plasma Seerum Seerum või plasma,
95800-9,Immuunglobuliini vabad kerged ahelad.paneel,-,-,U,-,,PANEL.CHEM,,,Uriin,
95966-8,Aspergillus glaucus antikehad.IgE,Acnc,Pt,S,Qn,,ALLERGY,,,Immuunglobuliin E Juhuslik Kvantitatiivne Seerum,
96043-5,Uratsüül,MCnc,Pt,S/P,Qn,,CHEM,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
96108-6,Klofasimiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96111-0,Linetsoliid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,"""

frBE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
103631-8,Natalizumab,Concentration de masse,Temps ponctuel,Sérum,Ordinal,IA,Médicaments et produits toxiques,,,,
106016-9,Bactéries,Présence ou identité,Temps ponctuel,Pénis,Nominal,Culture,Microbiologie,,,Verge,
106033-4,Bactéries,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture anaérobique,Microbiologie,,,,
106034-2,Champignon,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture,Microbiologie,,,,
103648-2,Hormone folliculo-stimulante^4 h post dose hormone de libération des gonadotrophines,Concentration arbitraire,Temps ponctuel,Sérum/Plasma,Quantitatif,,Tests de provocation,,,"4 h post dose GNRH FSH Gn-RF, Gonadotrophines-releasing factor",
103685-4,Citalopram,Concentration de masse,Temps ponctuel,Urine,Quantitatif,LC/MS/MS,Médicaments et produits toxiques,,,,
103806-6,Note,Observation,Temps ponctuel,Contact téléphonique,Document,Oncologie,DOC.CLINRPT,,,,
103830-6,Gabapentine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103834-8,Norbuprenorphine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103839-7,Phentermine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103958-5,Ofloxacine,Susceptibilité,Temps ponctuel,Isolat,Ordinal,Génotypage,Sensibilité aux antibiotiques,,,,
104133-4,Éthanol,PrThr,Temps ponctuel,Gaz expiré,Ordinal,,Médicaments et produits toxiques,,,,
104181-3,Toxine du clostridium tetani,PrThr,Temps ponctuel,Sérum/Plasma,Ordinal,Test biologique sur souris,Microbiologie,,,,
104183-9,Adénovirus ADN,PrThr,Temps ponctuel,Spécimen conjonctival,Ordinal,Sonde avec amplification de la cible,Microbiologie,,,,
104196-1,Créatine/Créatinine,Ratio de substance,Temps ponctuel,Sang sur papier filtre,Quantitatif,,Chimie,,,,
104234-0,Atomoxétine,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104237-3,Zopiclone,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104419-7,Legionella sp Ac^1er échantillon,Titre,Temps ponctuel,Sérum,Ordinal,IA,Microbiologie,,,Anticorps Echantillon.1,
104457-7,Virus varicelle-zona Anticorps.IgA,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,
104459-3,Virus varicelle-zona Anticorps.IgG,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,"""


In [299]:
### FIELD MAPPING DATASET - LOINCs ###

# This dictionary maps LOINC CSV fields to their corresponding OCL Concept fields.
# Mappings are based on the provided PDF and the LOINC FHIR example.
loinc_to_ocl_mapping = {    
    # General Format: '[Loinc Field]':'[OCL field]'

    #Field Mappings
    'LOINC_NUM': ['id','extras.Code_in_Source'],
    'LONG_COMMON_NAME': ['names.Fully-Specified.en[1]','extras.LONG_COMMON_NAME'],
    'DisplayName': ['names.Display.en[1]','extras.DisplayName'],
    'SHORTNAME': ['names.Short.en[1]','extras.SHORTNAME'],
    'CONSUMER_NAME': ['names.Consumer.en[1]','extras.CONSUMER_NAME'],
    'SCALE_TYP': ['datatype','extras.SCALE_TYP'],
    'STATUS': ['retired','extras.STATUS'], # Retired translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
    'DefinitionDescription': ['description','extras.DEFINITION_DESCRIPTION'],

    # Mappings derived from LOINC FHIR example JSON
    # These will be stored as `extras` to align with the FHIR properties.
    'COMPONENT': 'extras.COMPONENT',
    'PROPERTY': 'extras.PROPERTY',
    'TIME_ASPCT': 'extras.TIME_ASPCT',
    'SYSTEM': 'extras.SYSTEM',
    'METHOD_TYP': 'extras.METHOD_TYP',
    'VersionFirstReleased':'extras.VersionFirstReleased',
    'VersionLastChanged':'extras.VersionLastChanged',
    'ORDER_OBS':'extras.ORDER_OBS',
    'HL7_FIELD_SUBFIELD_ID':'extras.HL7_FIELD_SUBFIELD_ID',
    'EXTERNAL_COPYRIGHT_NOTICE':'extras.EXTERNAL_COPYRIGHT_NOTICE',
    'SURVEY_QUEST_TEXT':'extras.SURVEY_QUEST_TEXT',
    'SURVEY_QUEST_SRC':'extras.SURVEY_QUEST_SRC',
    'UNITSREQUIRED':'extras.UNITSREQUIRED',
    'RELATEDNAMES2':'extras.RELATEDNAMES2',
    'EXTERNAL_COPYRIGHT_LINK': 'extras.EXTERNAL_COPYRIGHT_LINK',
    'ValidHL7AttachmentRequest': 'extras.ValidHL7AttachmentRequest',
    'CHNG_TYPE': 'extras.CHNG_TYPE',
    'STATUS_TEXT': 'extras.STATUS_TEXT',
    'STATUS_REASON': 'extras.STATUS_REASON',
    'PanelType': 'extras.PanelType',
    'CHANGE_REASON_PUBLIC': 'extras.CHANGE_REASON_PUBLIC',
    'COMMON_TEST_RANK': 'extras.COMMON_TEST_RANK',
    'AskAtOrderEntry': 'extras.AskAtOrderEntry',
    'AssociatedObservations': 'extras.AssociatedObservations',
    'EXAMPLE_UNITS': 'extras.EXAMPLE_UNITS',
    'EXMPL_ANSWERS': 'extras.EXMPL_ANSWERS',
    'EXAMPLE_UCUM_UNITS': 'extras.EXAMPLE_UCUM_UNITS',
    'HL7_ATTACHMENT_STRUCTURE': 'extras.HL7_ATTACHMENT_STRUCTURE',
    'COMMON_ORDER_RANK': 'extras.COMMON_ORDER_RANK',
    'FORMULA': 'extras.FORMULA',
    'CLASS': 'extras.CLASS',
    'CLASSTYPE': 'extras.CLASSTYPE'

}

# This dictionary contains the fixed values to be used in the transformation.
values_loinc = {
    'type': 'Concept',
    'concept_class': 'LOINC',
    'source': 'LOINC',
    'owner_type': 'Organization',
    'owner': 'Regenstrief',
    'name_type_names.Fully-Specified.en[1]': 'Fully Specified',
    "locale_names.Fully-Specified.en[1]": "en",
    'locale_preferred_names.Fully-Specified.en[1]': True,
    'name_type_names.Short.en[1]': 'Short',
    "locale_names.Short.en[1]": "en",
    'locale_preferred_names.Short.en[1]': False,
    'name_type_names.Display.en[1]': 'Display',
    "locale_names.Display.en[1]": "en",
    'locale_preferred_names.Display.en[1]': False,
    'Extras.Code_Type': 'LOINC'
}

In [300]:
### FIELD MAPPING DATASET - LOINC Parts ###
loinc_part_to_ocl_mapping ={
    "PartNumber": ["id", "extras.Code_in_Source"],
    "PartTypeName": "extras.PartTypeName",
    "PartName": ["names.Fully-Specified", "extras.LONG_COMMON_NAME"],
    "PartDisplayName": ["names.Display", "extras.PartDisplayName"],
    'Status': ['retired','extras.STATUS'] # 'retired' translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
}

# This dictionary contains the fixed values to be used in the transformation.
fixed_values_loinc_parts = {
    'type': 'Concept',
    'concept_class': 'LOINC Part',
    'datatype': 'N/A',
    'source': 'LOINC',
    'owner_type': 'Organization',
    'owner': 'Regenstrief',
    'name_type_names.Fully-Specified.en[1]': 'Fully Specified',
    "locale_names.Fully-Specified.en[1]": "en",
    'locale_preferred_names.Fully-Specified.en[1]': True,
    'name_type_names.Display.en[1]': 'Display',
    "locale_names.Display.en[1]": "en",
    'locale_preferred_names.Display.en[1]': False,
    'Extras.Code_Type': 'LOINC Part'
}

In [301]:
### FIELD MAPPING DATASET - LOINC Answer Lists and Answers ###

# Answer Lists
answer_list_to_ocl_mapping = {
    "AnswerListId": ["id", "extras.Code_in_Source"],
    "AnswerListName": ["names.Fully-Specified", "extras.AnswerListName"],
    "AnswerListOID": "extras.AnswerListOID",
    "ExtDefinedYN": "extras.ExtDefinedYN",
    "ExtDefinedAnswerListCodeSystem": "extras.ExtDefinedAnswerListCodeSystem",
    "ExtDefinedAnswerListLink": "extras.ExtDefinedAnswerListLink"
}

fixed_values_answer_list = {
    'type': 'Concept',
    "concept_class": "Answer List",
    "datatype": "N/A",
    "source": "LOINC",
    "owner_type": "Organization",
    "owner": "Regenstrief",
    "name_type_names.Fully-Specified.en[1]": "Fully Specified",
    "locale_names.Fully-Specified.en[1]": "en",
    "locale_preferred_names.Fully-Specified.en[1]": True,
    "Extras.Code_Type": "LOINC Answer List"
}

# Answers
answer_to_ocl_mapping = {
    "AnswerStringId": ["id", "extras.Code_in_Source"],
    "DisplayText": ["names.Display", "extras.DisplayText"],
    "LocalAnswerCode": "extras.LocalAnswerCode",
    "LocalAnswerCodeSystem": "extras.LocalAnswerCodeSystem",
    "SequenceNumber": "extras.SequenceNumber",
    "ExtCodeId": "extras.ExtCodeId",
    "ExtCodeDisplayName": ["names.Fully-Specified", "extras.ExtCodeDisplayName"],
    "ExtCodeSystem": "extras.ExtCodeSystem",
    "ExtCodeSystemVersion": "extras.ExtCodeSystemVersion",
    "ExtCodeSystemCopyrightNotice": "extras.ExtCodeSystemCopyrightNotice",
    "SubsequentTextPrompt": "extras.SubsequentTextPrompt",
    "Description": "extras.Description",
    "Score": "extras.Score"
}

fixed_values_answer = {
    'type': 'Concept',
    "concept_class": "LOINC Answer",
    "datatype": "N/A",
    "source": "LOINC",
    "owner_type": "Organization",
    "owner": "Regenstrief",
    "name_type_names.Display.en[1]": "Display",
    "locale_preferred_names.Display.en[1]": True,
    "locale_names.Display.en[1]": "en",
    "name_type_names.Fully-Specified.en[1]": "Fully-Specified",
    "locale_preferred_names.Fully-Specified.en[1]": False,
    "locale_names.Fully-Specified.en[1]": "en",
    "Extras.Code_Type": "LOINC Answer"
}

In [302]:
#Transformation rules - specify fields to transform and their target values

transformation_rules =  [
    {
      "field": "STATUS",
      "transformations": {
        "DEPRECATED": True,
        "ACTIVE": False,
        "TRIAL": False,
        "DISCOURAGED": False
      },
      "target_field": "retired"
    }
  ]

## Phase 1: Data Loading and Validation

In [303]:
# Data Loading and Validation - Concepts

def load_data():
    """Loads data based on the `mode` variable and validates fields against the mapping."""
    if mode == 2:
        loinc_df = demo_loinc_df
        part_link_df = demo_part_link_df
        answer_list_df = demo_answer_list_df
    elif mode == 1:
        # In test mode, we load a small subset of the real data
        try:
            loinc_df = pd.read_csv(loinc_csv_path, nrows=15, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")
            
        part_link_df = pd.read_csv(part_link_csv_path, nrows=15, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, nrows=15, low_memory=False)
    elif mode == 0:
        # In full run mode, we load the entire CSV files
        try:
            loinc_df = pd.read_csv(loinc_csv_path, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")
            
        part_link_df = pd.read_csv(part_link_csv_path, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, low_memory=False)
    else:
        print("Invalid mode selected. Please use 0, 1, or 2.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    # ... (rest of the load_data function remains the same) ...
    # The validation logic below is not changed.

    # A helper function to create a DataFrame from the raw CSV data
    def load_csv_data(csv_data):
        return pd.read_csv(StringIO(csv_data))

    # This part needs to be changed. The code below should dynamically
    # load files from the linguistic_variants_path directory for modes 0 and 1.
    if mode in [0, 1]:
        print("Loading linguistic variant files...")
        linguistic_variant_dfs = []
        for filename in os.listdir(linguistic_variants_path):
            if filename.endswith('.csv'):
                filepath = os.path.join(linguistic_variants_path, filename)
                df = pd.read_csv(filepath, low_memory=False)
                linguistic_variant_dfs.append(df)
    else: # mode == 2
        # Use hardcoded data for demo mode
        esMX_df = load_csv_data(esMX_data)
        etEE_df = load_csv_data(etEE_data)
        frBE_df = load_csv_data(frBE_data)
        linguistic_variant_dfs = [esMX_df, etEE_df, frBE_df]
    
    print(f"Loaded {len(linguistic_variant_dfs)} linguistic variant files.")
    
    return loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs

# Call the function to load all data
loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs = load_data()

Loading linguistic variant files...
Loaded 3 linguistic variant files.


In [304]:
# Adds linguistic variant file names, dynamically generated from the directory.

linguistic_variant_files = [os.path.join(linguistic_variants_path, f) for f in os.listdir(linguistic_variants_path) if f.endswith('.csv')]

# Function to create the Fully Specified Name (FSN)
def create_fsn(row):
    parts = [row['COMPONENT'], row['PROPERTY'], row['TIME_ASPCT'], row['SYSTEM'], row['SCALE_TYP'], row['METHOD_TYP'], row['CLASS']]
    return ':'.join([str(part) for part in parts if pd.notna(part) and str(part).strip() != ''])

# Function to process a single linguistic variant file and add the locale code
def process_linguistic_file(file_name):
    locale_code = os.path.basename(file_name)[:2]
    df = pd.read_csv(file_name, low_memory=False)
    processed_rows = []
    
    for index, row in df.iterrows():
        fsn = create_fsn(row)
        if fsn:
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': fsn, 'NameType': 'Fully-Specified'
            })
        if pd.notna(row['SHORTNAME']) and row['SHORTNAME'].strip() != '':
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': row['SHORTNAME'], 'NameType': 'Short'
            })
        if pd.notna(row['LONG_COMMON_NAME']) and row['LONG_COMMON_NAME'].strip() != '':
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': row['LONG_COMMON_NAME'], 'NameType': 'Display'
            })
        if pd.notna(row['RELATEDNAMES2']) and row['RELATEDNAMES2'].strip() != '':
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': row['RELATEDNAMES2'], 'NameType': 'None'
            })
            
    return pd.DataFrame(processed_rows)

# Process all linguistic variant files and concatenate the results
all_processed_variants = pd.concat(
    [process_linguistic_file(file) for file in linguistic_variant_files], 
    ignore_index=True
)


# Create a copy to avoid modifying the original dataframe
temp_df = all_processed_variants.copy()

# Sort the data to ensure the numeric counter is assigned consistently.
temp_df.sort_values(by=['LOINC_NUM', 'NameType', 'LocaleCode'], inplace=True)

# Generate a numeric counter for each unique combination of LOINC_NUM, NameType, and LocaleCode.
temp_df['counter'] = temp_df.groupby(['LOINC_NUM', 'NameType', 'LocaleCode']).cumcount() + 1

# Create a new column that will be used for the pivot operation.
# The format will be 'names.[NameType].[LocaleCode][counter]'.
temp_df['pivot_column'] = 'names.' + temp_df['NameType'].astype(str) + '.' + temp_df['LocaleCode'].astype(str) + '[' + temp_df['counter'].astype(str) + ']'

# Pivot the DataFrame using the new 'pivot_column' and 'LinguisticVariantName' values.
pivoted_variants = temp_df.pivot(index='LOINC_NUM', columns='pivot_column', values='LinguisticVariantName')

# Flatten the MultiIndex columns and reset the index.
pivoted_variants.columns = pivoted_variants.columns.get_level_values(0)
pivoted_variants.reset_index(inplace=True)

# Merge the pivoted variants with the main LOINC DataFrame.
merged_loinc_df = pd.merge(loinc_df, pivoted_variants, on='LOINC_NUM', how='left')

# To see the pivoted result, you can display the head of the new dataframe
# print("\nPivoted Variants DataFrame head with new linguistic variant columns:")
# print(pivoted_variants.head())

## Phase 2: Concept Creation

In this phase, we will transform LOINC terms, parts, answer lists, and linguistic variants into OCL Concept objects.

## Phase 3: Mapping Creation

This phase is for creating OCL Mapping objects based on relational files like `PanelsAndForms.csv` and `MapTo.csv`.

In [ ]:
# Your code to create OCL Mapping objects goes here.


## Phase 4: Hierarchy Creation

This phase focuses on parsing the `ComponentHierarchyBySystem.csv` file and establishing hierarchy relationships between concepts.

In [ ]:
# Your code to build the hierarchy goes here.


## Phase 5: UMLS Enhancement

Here, we'll query an external UMLS API to enrich the concepts with CUIs (Concept Unique Identifiers).

In [ ]:
# Your code to query the UMLS API and add CUIs goes here.


## Phase 6: Output Generation

The final phase is to save all the created OCL objects to a JSON Lines file (`.jsonl`) for import into the OCL system.

In [ ]:
# Your code to format and save the final output goes here.
